In [1]:
# RUN THIS CELL FIRST TO IMPORT ALL NECESSARY LIBRARIES
import pandas as pd
import numpy as np
import requests
import time
import io
import os

### EXTRACT DATA

In [ ]:
# RUN THIS CELL ONLY TO UPDATE THE ISTAT DATA
# this cell save the istat data in a raw data folder in order to avoid running the api call every time
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
header = {"Accept":"application/vnd.sdmx.data+csv;version=1.0.0"}
response = requests.get(url, headers=header)
print(response.status_code)

with open("data/raw/istat_incidenti.csv", "w", encoding="utf-8") as f:
    f.write(response.text)

200


In [30]:
# RUN THIS CELL ONCE TO DOWNLOAD SITUAS DATA
# this cell save the situas data from 31-12-2001 to 31-12-2024 in 24 different csv files
date_ranges = [f"{year}-12-31" for year in range(2001, 2025)]

# to download files in csv format is necessary to set json and header parameters.
# You can acknowledge this by using dev tools on the web page at the below specified url.
json_situas = {"orderFields":[],"orderDirects":[],"pFilterFields":[],"pFilterValues":[]}
header_situas = {"Content-Type": "application/json-patch+json"}

for i in date_ranges:
    url_situas = f"https://situas.istat.it/ShibO2Module/api/Report/Spool/{i}/74?&pdoctype=CSV"
    time.sleep(1)
    response_situas = requests.post(url_situas, json = json_situas, headers = header_situas)
    print(response_situas.status_code)
    if response_situas.status_code == 200:
        with open(f"data/raw/situas_{i[0:4]}.csv", "w", encoding="utf-8") as f:
            f.write(response_situas.text)
    else:
        # if an http status code different from 200 is encounterd, this make sure that cell execution is stopped
        raise SystemExit(f"Abort due to http error {response_situas.status_code}")
print(f"All done - {len(date_ranges)} files downloaded")


200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
All done - 24 files downloaded


### CLEAN DATA

In [42]:
situas_df = pd.read_csv("data/raw/situas_2001.csv", sep = ";", decimal = ",")
situas_df["Anno"] = 2001
print(f"Year 2001:{situas_df.shape}")
situas_df = [situas_df]

for i in range(2002,2025):
    tmp_df = pd.read_csv(f"data/raw/situas_{i}.csv", sep = ";", decimal = ",")
    tmp_df["Anno"] = i # add a column with the year of the data

# Analyzing the csv situas data, it can be noticed that from 2015 on there is another column 
# not present in the previous years "Codice Provincia (Storico)"". This is simply the code that 
# in the previous years is called "Codice Provincia/Uts" which from 2015 on contains different values. 
# So, we can drop "Codice Provincia/Uts" and change "Codice Provincia (Storico)"" name in "Codice Provincia/Uts"
# in order to obtain a uniform dataframe.
    if i in range(2015,2025):
        tmp_df.drop(columns=["Codice Provincia/Uts"], inplace = True)
        tmp_df.rename(columns={"Codice Provincia (Storico)": "Codice Provincia/Uts"}, inplace = True)

    # check if columns name are the same among csv files in both directions
    check_column_names_sx = set(situas_df[0].columns) - set(tmp_df.columns)
    check_column_names_dx = set(tmp_df.columns) - set(situas_df[0].columns)

    # check if columns order are the same among csv files
    check_column_order = list(situas_df[0].columns) == list(tmp_df.columns)

    if len(check_column_names_sx) == 0 and len(check_column_names_dx) == 0 and check_column_order:
        print(f"Year {i}: {tmp_df.shape}")
        situas_df.append(tmp_df)
    else:
        print(situas_df[0].columns)
        print(tmp_df.columns)
        raise SystemExit(f"Abort due to column name mismatch in situas_{i}.csv")
situas_df = pd.concat(situas_df, ignore_index = True)
print(situas_df.sample(10))



Year 2001:(8102, 17)
Year 2002: (8102, 17)
Year 2003: (8100, 17)
Year 2004: (8101, 17)
Year 2005: (8101, 17)
Year 2006: (8101, 17)
Year 2007: (8101, 17)
Year 2008: (8101, 17)
Year 2009: (8100, 17)
Year 2010: (8094, 17)
Year 2011: (8092, 17)
Year 2012: (8092, 17)
Year 2013: (8090, 17)
Year 2014: (8057, 17)
Year 2015: (8046, 17)
Year 2016: (7998, 17)
Year 2017: (7978, 17)
Year 2018: (7954, 17)
Year 2019: (7914, 17)
Year 2020: (7903, 17)
Year 2021: (7904, 17)
Year 2022: (7904, 17)
Year 2023: (7900, 17)
Year 2024: (7896, 17)
        ï»¿Codice Ripartizione geografica  Codice Regione  \
161635                                  1               1   
171593                                  1               3   
180358                                  2               5   
161338                                  1               1   
185103                                  1               1   
50585                                   1               3   
39086                                   4     

From the shapes of the previous cell, we can notice that the number of rows change for every situas csv file. Since that every record in these csv files is a different town, this means that the number of town changes every year. This happens because some towns splits or merge toghether with others. This kind of town usually are very small so it's difficult that they could be interesting for investment from business. So we can simply keep just the towns that are present in every csv file. In order to confirm this strategy, we can calculate how many towns we will exclude from the analysis.

In [43]:
comuni_per_anno = situas_df.groupby("Anno")["Codice Comune (numerico)"].apply(set)  # o come si chiama la colonna anno/codice da voi
comuni_comuni_a_tutti = set.intersection(*comuni_per_anno)
print(len(comuni_comuni_a_tutti))

7488
